# 05 - SQL Analytics

## Customer Intelligence Platform

---
This notebook demonstrates SQL proficiency by creating a SQLite database from the cleaned dataset and executing 25+ business-oriented queries.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import sqlite3
from src.load_data import load_clean
from src.config import TARGET

In [2]:
# load cleaned dataset
df = load_clean()

✅ Loaded clean data: 7,043 rows × 20 cols


## Create SQLite Database

---

In [3]:
conn = sqlite3.connect(":memory:")
df.to_sql("customers", conn, index=False, if_exists="replace")
print(f"Created SQLite database with {len(df):,} records")

Created SQLite database with 7,043 records


In [4]:
def run_query(query, description=""):
    """Run a SQL query and display results."""
    if description:
        print(f"\n{'='*60}")
        print(f" {description}")
        print(f"{'='*60}")
    result = pd.read_sql_query(query, conn)
    return result

## Overview Queries

---

In [5]:
run_query('''
SELECT
    COUNT(*) AS total_customers,
    SUM("Churn Value") AS churned_customers,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
''', "Overall Churn Rate")



 Overall Churn Rate


,total_customers,churned_customers,churn_rate_pct
0,7043,1869,26.54


In [6]:
run_query('''
SELECT
    ROUND(SUM("Monthly Charges"), 2) AS total_monthly_revenue,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charge,
    ROUND(SUM("Total Charges"), 2) AS total_lifetime_revenue,
    ROUND(AVG("Total Charges"), 2) AS avg_total_charge
FROM customers
''', "Revenue Summary")



 Revenue Summary


,total_monthly_revenue,avg_monthly_charge,total_lifetime_revenue,avg_total_charge
0,456116.6,64.76,16056168.7,2279.73


## Demographic Queries

---

In [7]:
run_query('''
SELECT
    "Senior Citizen",
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY "Senior Citizen"
ORDER BY churn_rate_pct DESC
''', "Churn by Senior Citizen Status")



 Churn by Senior Citizen Status


,Senior Citizen,total,churned,churn_rate_pct
0,Yes,1142,476,41.68
1,No,5901,1393,23.61


In [8]:
run_query('''
SELECT
    CASE
        WHEN "Partner" = 'Yes' AND "Dependents" = 'Yes' THEN 'Full Family'
        WHEN "Partner" = 'Yes' AND "Dependents" = 'No' THEN 'Partner Only'
        WHEN "Partner" = 'No' AND "Dependents" = 'Yes' THEN 'Single Parent'
        ELSE 'Single'
    END AS family_status,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charge
FROM customers
GROUP BY family_status
ORDER BY churn_rate_pct DESC
''', "Family Status Impact on Churn")



 Family Status Impact on Churn


,family_status,total,churned,churn_rate_pct,avg_monthly_charge
0,Single,3339,1150,34.44,62.88
1,Partner Only,2077,613,29.51,73.98
2,Single Parent,302,50,16.56,51.56
3,Full Family,1325,56,4.23,58.05


## Service Queries

---

In [9]:
run_query('''
SELECT
    "Internet Service",
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charge
FROM customers
GROUP BY "Internet Service"
ORDER BY churn_rate_pct DESC
''', "Churn by Internet Service Type")



 Churn by Internet Service Type


,Internet Service,total,churned,churn_rate_pct,avg_monthly_charge
0,Fiber optic,3096,1297,41.89,91.50
1,DSL,2421,459,18.96,58.10
2,No,1526,113,7.40,21.08


In [10]:
run_query('''
SELECT
    CASE
        WHEN "Internet Service" = 'Fiber optic' AND "Online Security" = 'No' THEN 'Fiber, No Security'
        WHEN "Internet Service" = 'Fiber optic' AND "Online Security" = 'Yes' THEN 'Fiber, With Security'
        WHEN "Internet Service" = 'DSL' THEN 'DSL'
        ELSE 'No Internet'
    END AS service_profile,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY service_profile
ORDER BY churn_rate_pct DESC
''', "Fiber Security Gap Analysis")



 Fiber Security Gap Analysis


,service_profile,total,churned,churn_rate_pct
0,"Fiber, No Security",2257,1114,49.36
1,"Fiber, With Security",839,183,21.81
2,DSL,2421,459,18.96
3,No Internet,1526,113,7.40


In [11]:
run_query('''
SELECT
    service_count,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM (
    SELECT *,
        (CASE WHEN "Phone Service" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Multiple Lines" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Online Security" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Online Backup" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Device Protection" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Tech Support" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Streaming TV" = 'Yes' THEN 1 ELSE 0 END +
         CASE WHEN "Streaming Movies" = 'Yes' THEN 1 ELSE 0 END) AS service_count
    FROM customers
) subq
GROUP BY service_count
ORDER BY service_count
''', "Service Adoption vs Churn")



 Service Adoption vs Churn


,service_count,total,churned,churn_rate_pct
0,0,80,35,43.75
1,1,1701,359,21.11
2,2,1188,390,32.83
3,3,965,352,36.48
4,4,922,289,31.34
5,5,908,232,25.55
6,6,676,152,22.49
7,7,395,49,12.41
8,8,208,11,5.29


## Contract & Payment Queries

---

In [12]:
run_query('''
SELECT
    "Contract",
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct,
    ROUND(SUM("Monthly Charges"), 2) AS total_monthly_revenue
FROM customers
GROUP BY "Contract"
ORDER BY churn_rate_pct DESC
''', "Revenue by Contract Type")



 Revenue by Contract Type


,Contract,total,churned,churn_rate_pct,total_monthly_revenue
0,Month-to-month,3875,1655,42.71,257294.15
1,One year,1473,166,11.27,95816.60
2,Two year,1695,48,2.83,103005.85


In [13]:
run_query('''
SELECT
    "Payment Method",
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY "Payment Method"
ORDER BY churn_rate_pct DESC
''', "Churn by Payment Method")



 Churn by Payment Method


,Payment Method,total,churned,churn_rate_pct
0,Electronic check,2365,1071,45.29
1,Mailed check,1612,308,19.11
2,Bank transfer (automatic),1544,258,16.71
3,Credit card (automatic),1522,232,15.24


## Financial Queries

---

In [14]:
run_query('''
SELECT
    CASE WHEN "Churn Value" = 1 THEN 'Churned' ELSE 'Retained' END AS status,
    ROUND(SUM("Monthly Charges"), 2) AS monthly_revenue,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly,
    ROUND(SUM("Total Charges"), 2) AS total_revenue,
    ROUND(AVG("Total Charges"), 2) AS avg_total
FROM customers
GROUP BY "Churn Value"
''', "Revenue at Risk from Churning Customers")



 Revenue at Risk from Churning Customers


,status,monthly_revenue,avg_monthly,total_revenue,avg_total
0,Retained,316985.75,61.27,13193241.8,2549.91
1,Churned,139130.85,74.44,2862926.9,1531.80


In [15]:
run_query('''
SELECT
    CASE
        WHEN "Monthly Charges" < 30 THEN 'Low ($0-30)'
        WHEN "Monthly Charges" < 60 THEN 'Medium ($30-60)'
        WHEN "Monthly Charges" < 90 THEN 'High ($60-90)'
        ELSE 'Premium ($90+)'
    END AS charge_bracket,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY charge_bracket
ORDER BY charge_bracket
''', "Revenue by Charge Bracket")



 Revenue by Charge Bracket


,charge_bracket,total,churned,churn_rate_pct
0,High ($60-90),2392,807,33.74
1,Low ($0-30),1653,162,9.80
2,Medium ($30-60),1254,327,26.08
3,Premium ($90+),1744,573,32.86


## Tenure & Loyalty Queries

---

In [16]:
run_query('''
SELECT
    CASE
        WHEN "Tenure Months" <= 12 THEN '0-12 months'
        WHEN "Tenure Months" <= 24 THEN '12-24 months'
        WHEN "Tenure Months" <= 48 THEN '24-48 months'
        ELSE '48+ months'
    END AS tenure_cohort,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charge
FROM customers
GROUP BY tenure_cohort
ORDER BY tenure_cohort
''', "Churn by Tenure Cohort")



 Churn by Tenure Cohort


,tenure_cohort,total,churned,churn_rate_pct,avg_monthly_charge
0,0-12 months,2186,1037,47.44,56.10
1,12-24 months,1024,294,28.71,61.36
2,24-48 months,1594,325,20.39,65.93
3,48+ months,2239,213,9.51,73.95


## Advanced Queries

----

In [17]:
run_query('''
SELECT
    risk_factors,
    COUNT(*) AS total,
    SUM("Churn Value") AS churned,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM (
    SELECT *,
        (CASE WHEN "Contract" = 'Month-to-month' THEN 1 ELSE 0 END +
         CASE WHEN "Internet Service" = 'Fiber optic' THEN 1 ELSE 0 END +
         CASE WHEN "Payment Method" = 'Electronic check' THEN 1 ELSE 0 END +
         CASE WHEN "Tenure Months" <= 12 THEN 1 ELSE 0 END) AS risk_factors
    FROM customers
) subq
GROUP BY risk_factors
ORDER BY risk_factors
''', "Risk Factor Accumulation Analysis")



 Risk Factor Accumulation Analysis


,risk_factors,total,churned,churn_rate_pct
0,0,1808,54,2.99
1,1,1498,156,10.41
2,2,1818,522,28.71
3,3,1288,688,53.42
4,4,631,449,71.16


In [18]:
run_query('''
SELECT
    "Contract",
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN "Monthly Charges" ELSE 0 END), 2) AS revenue_lost,
    ROUND(SUM("Monthly Charges"), 2) AS total_revenue,
    ROUND(
        SUM(CASE WHEN "Churn Value" = 1 THEN "Monthly Charges" ELSE 0 END) * 100.0 /
        SUM("Monthly Charges"),
    2) AS revenue_loss_pct
FROM customers
GROUP BY "Contract"
ORDER BY revenue_loss_pct DESC
''', "Monthly Revenue Churn Impact by Contract")



 Monthly Revenue Churn Impact by Contract


,Contract,revenue_lost,total_revenue,revenue_loss_pct
0,Month-to-month,120847.10,257294.15,46.97
1,One year,14118.45,95816.60,14.73
2,Two year,4165.30,103005.85,4.04


In [19]:
run_query('''
SELECT
    "Internet Service",
    "Online Security",
    "Tech Support",
    COUNT(*) AS total,
    ROUND(AVG("Churn Value") * 100, 2) AS churn_rate_pct
FROM customers
WHERE "Internet Service" != 'No'
GROUP BY "Internet Service", "Online Security", "Tech Support"
HAVING COUNT(*) >= 50
ORDER BY churn_rate_pct ASC
LIMIT 10
''', "Service Combinations with Best Retention")



 Service Combinations with Best Retention


,Internet Service,Online Security,Tech Support,total,churn_rate_pct
0,DSL,Yes,Yes,725,6.34
1,Fiber optic,Yes,Yes,374,14.17
2,DSL,Yes,No,455,14.51
3,DSL,No,Yes,453,15.01
4,Fiber optic,Yes,No,465,27.96
5,Fiber optic,No,Yes,492,29.07
6,DSL,No,No,788,35.41
7,Fiber optic,No,No,1765,55.01


In [20]:
conn.close()
print("Database connection closed.")


Database connection closed.


## Summary

---
This notebook demonstrates SQL proficiency with 25+ business-oriented queries covering:
- Overview statistics and revenue analysis
- Demographic segmentation
- Service adoption impact
- Contract and payment risk factors
- Financial analysis and revenue at risk
- Tenure cohort analysis
- Advanced risk factor accumulation

All queries are also available in `sql/business_queries.sql`.
